# U.S. Drought Monitor — Weekly Drought Status by County

**What it is.** The authoritative weekly U.S. drought map (NDMC / USDA / NOAA). For any
county you get the **percent of area** in each drought category **D0** (abnormally dry) →
**D4** (exceptional drought), every week back to 2000.

| | |
|---|---|
| **Coverage** | United States — county / state / national |
| **History** | Weekly, 2000 → present |
| **Cost / key** | **Free · no key** (REST, JSON/CSV/XML) |
| **Docs** | https://droughtmonitor.unl.edu/DmData/DataDownload/WebServiceInfo.aspx |

**Why it's core to Tracera.** A clean, official drought signal to correlate against yield and
insurance losses, and to flag in-season water stress.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test + core query — weekly drought % for our county

In [2]:
USDM = "https://usdmdataservices.unl.edu/api"

def drought_county(fips=FIPS, start="1/1/2012", end="12/31/2024"):
    r = requests.get(f"{USDM}/CountyStatistics/GetDroughtSeverityStatisticsByAreaPercent",
                     params={"aoi": fips, "startdate": start, "enddate": end, "statisticsType": "1"},
                     headers={"Accept": "application/json"}, timeout=90)
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df["mapDate"] = pd.to_datetime(df["mapDate"])
    for c in ("none", "d0", "d1", "d2", "d3", "d4"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.sort_values("mapDate").reset_index(drop=True)

usdm = drought_county()
assert not usdm.empty
print(f"OK — {len(usdm)} weekly records for {COUNTY} County")
usdm[["mapDate", "none", "d0", "d1", "d2", "d3", "d4"]].tail(6)

OK — 680 weekly records for STORY County


,mapDate,none,d0,d1,d2,d3,d4
674,2024-11-26,0.0,100.0,0.0,0.0,0.0,0.0
675,2024-12-03,0.0,100.0,0.0,0.0,0.0,0.0
676,2024-12-10,0.0,100.0,0.0,0.0,0.0,0.0
677,2024-12-17,0.0,100.0,0.0,0.0,0.0,0.0
678,2024-12-24,0.0,100.0,0.0,0.0,0.0,0.0
679,2024-12-31,0.0,100.0,0.0,0.0,0.0,0.0


### 2. Derived signal — Drought Severity & Coverage Index (DSCI, 0–500)

In [3]:
# statisticsType=1 gives CUMULATIVE area % (d0 = "D0 or worse"), so DSCI is the simple
# sum of d0..d4 and ranges 0 (no drought) to 500 (100% of area in D4).
usdm["DSCI"] = usdm[["d0", "d1", "d2", "d3", "d4"]].sum(axis=1)
annual = usdm.assign(year=usdm.mapDate.dt.year).groupby("year")["DSCI"].mean().round(1)
print("Mean annual DSCI (higher = more drought):")
annual.tail(10)

Mean annual DSCI (higher = more drought):


year
2015      0.0
2016     19.9
2017     55.9
2018     14.7
2019     16.4
2020     93.7
2021    169.0
2022     99.8
2023    124.8
2024    133.6
Name: DSCI, dtype: float64

### Notes & how to extend
- **Endpoints:** swap `CountyStatistics` for `StateStatistics` / `USStatistics`; add
  `GetNonConsecutiveStatisticsCounty` for "weeks in drought over threshold".
- **`statisticsType`:** 1 = cumulative area %, 2 = categorical (non-overlapping) area %.
- **DSCI** is the standard way to reduce D0–D4 to one comparable weekly number for modeling.
- Pair with **RMA Cause of Loss** (drought indemnities) and **QuickStats** (yield) to quantify
  drought's local yield impact.